# intersect_qef old/ref/new JIT comparison

This notebook JIT-compiles temporary bindings for `intersect_qef_old`, `intersect_qef_ref`, `intersect_qef`, and a CPU-only `intersect_qef` wrapper. It then uses `notebooks/test.glb` to compare occupancy, QEF outputs, timing, and CUDA peak memory.

Run this notebook from the `symm-enforce` conda environment.

In [1]:
import os
import sys
import time
import tempfile
import statistics
from pathlib import Path

import torch
from torch.utils.cpp_extension import load

ROOT = Path.cwd()
if not (ROOT / 'setup.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'setup.py').exists(), f'Cannot find repo root from {Path.cwd()}'

os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)
torch.set_grad_enabled(False)

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

# GRID_SIZE = int(os.environ.get('FDG_GRID_SIZE', '128'))
GRID_SIZE = 512
CHUNK_TRIANGLES = int(os.environ.get('FDG_CHUNK_TRIANGLES', '20000'))
SUBDIVIDE_AREA_THRESHOLD = float(os.environ.get('FDG_SUBDIVIDE_AREA_THRESHOLD', str(1.0 / (GRID_SIZE * GRID_SIZE))))
SUBDIVIDE_MAX_ITERS = int(os.environ.get('FDG_SUBDIVIDE_MAX_ITERS', '12'))

print('repo:', ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('MAX_JOBS:', os.environ['MAX_JOBS'])
print('GRID_SIZE:', GRID_SIZE, 'CHUNK_TRIANGLES:', CHUNK_TRIANGLES)
print('SUBDIVIDE_AREA_THRESHOLD:', SUBDIVIDE_AREA_THRESHOLD, 'SUBDIVIDE_MAX_ITERS:', SUBDIVIDE_MAX_ITERS)


repo: /mnt/nvmefs/Projects/o-voxel-gpu
python: /home/quanta/.conda/envs/symm-enforce/bin/python
torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
MAX_JOBS: 32
GRID_SIZE: 512 CHUNK_TRIANGLES: 20000
SUBDIVIDE_AREA_THRESHOLD: 3.814697265625e-06 SUBDIVIDE_MAX_ITERS: 12


## JIT compile temporary extension

The binding source is generated inside this notebook into a temporary directory. No project source file is modified here.

In [2]:
build_dir = Path(tempfile.mkdtemp(prefix='fdg_intersect_jit_'))
binding_path = build_dir / 'binding.cpp'

binding_cpp = f'''
#include <torch/extension.h>
#include <Eigen/Dense>
#include <unordered_map>
#include <vector>
#include <cmath>
#include "{(ROOT / 'src/convert/flexible_dual_grid.cpp').as_posix()}"
#include "{(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/api.h').as_posix()}"

namespace py = pybind11;

namespace {{

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor>
intersect_qef_cpu_only(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    int64_t chunk_triangles) {{
    (void)chunk_triangles;
    const int64_t T = triangles.size(0);
    const float* tri_ptr = triangles.data_ptr<float>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();

    std::vector<Eigen::Vector3f> tri_vec;
    tri_vec.reserve(static_cast<size_t>(T) * 3);
    for (int64_t t = 0; t < T; ++t) {{
        tri_vec.emplace_back(tri_ptr[9 * t + 0], tri_ptr[9 * t + 1], tri_ptr[9 * t + 2]);
        tri_vec.emplace_back(tri_ptr[9 * t + 3], tri_ptr[9 * t + 4], tri_ptr[9 * t + 5]);
        tri_vec.emplace_back(tri_ptr[9 * t + 6], tri_ptr[9 * t + 7], tri_ptr[9 * t + 8]);
    }}

    std::unordered_map<VoxelCoord, size_t> hash_table;
    std::vector<int3> voxels;
    std::vector<Eigen::Vector3f> means;
    std::vector<float> cnt;
    std::vector<bool3> intersected;
    std::vector<Eigen::Matrix4f> qefs;
    intersect_qef(
        Eigen::Vector3f(vs[0], vs[1], vs[2]),
        Eigen::Vector3i(gr[0], gr[1], gr[2]),
        Eigen::Vector3i(gr[3], gr[4], gr[5]),
        tri_vec,
        hash_table,
        voxels,
        means,
        cnt,
        intersected,
        qefs);

    const int64_t N = static_cast<int64_t>(voxels.size());
    auto opts_i32 = torch::TensorOptions().dtype(torch::kInt32).device(torch::kCPU);
    auto opts_f32 = torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU);
    auto opts_bool = torch::TensorOptions().dtype(torch::kBool).device(torch::kCPU);
    auto out_voxels = torch::empty({{N, 3}}, opts_i32);
    auto out_mean = torch::empty({{N, 3}}, opts_f32);
    auto out_cnt = torch::empty({{N}}, opts_f32);
    auto out_intersected = torch::empty({{N, 3}}, opts_bool);
    auto out_qefs = torch::empty({{N, 10}}, opts_f32);

    int32_t* vox_ptr = out_voxels.data_ptr<int32_t>();
    float* mean_ptr = out_mean.data_ptr<float>();
    float* cnt_ptr = out_cnt.data_ptr<float>();
    bool* inter_ptr = out_intersected.data_ptr<bool>();
    float* qef_ptr = out_qefs.data_ptr<float>();
    for (int64_t i = 0; i < N; ++i) {{
        vox_ptr[3 * i + 0] = voxels[i].x;
        vox_ptr[3 * i + 1] = voxels[i].y;
        vox_ptr[3 * i + 2] = voxels[i].z;
        mean_ptr[3 * i + 0] = means[i].x();
        mean_ptr[3 * i + 1] = means[i].y();
        mean_ptr[3 * i + 2] = means[i].z();
        cnt_ptr[i] = cnt[i];
        inter_ptr[3 * i + 0] = intersected[i].x;
        inter_ptr[3 * i + 1] = intersected[i].y;
        inter_ptr[3 * i + 2] = intersected[i].z;
        const Eigen::Matrix4f& Q = qefs[i];
        qef_ptr[10 * i + 0] = Q(0, 0);
        qef_ptr[10 * i + 1] = Q(0, 1);
        qef_ptr[10 * i + 2] = Q(0, 2);
        qef_ptr[10 * i + 3] = Q(0, 3);
        qef_ptr[10 * i + 4] = Q(1, 1);
        qef_ptr[10 * i + 5] = Q(1, 2);
        qef_ptr[10 * i + 6] = Q(1, 3);
        qef_ptr[10 * i + 7] = Q(2, 2);
        qef_ptr[10 * i + 8] = Q(2, 3);
        qef_ptr[10 * i + 9] = Q(3, 3);
    }}
    return std::make_tuple(out_voxels, out_mean, out_cnt, out_intersected, out_qefs);
}}

torch::Tensor intersect_occ_cpu_only(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    int64_t chunk_triangles) {{
    return std::get<0>(intersect_qef_cpu_only(triangles, voxel_size, grid_range, chunk_triangles));
}}

}} // namespace

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {{
    m.def("intersect_qef_cpu", &intersect_qef_cpu_only);
    m.def("intersect_occ_cpu", &intersect_occ_cpu_only);
    m.def("intersect_qef_old", &o_voxel::fdg::intersect_qef_old);
    m.def("intersect_occ_old", &o_voxel::fdg::intersect_occ_old);
    m.def("intersect_qef_ref", &o_voxel::fdg::intersect_qef_ref);
    m.def("intersect_qef", &o_voxel::fdg::intersect_qef);
    m.def("intersect_occ", &o_voxel::fdg::intersect_occ);
}}
'''

binding_path.write_text(binding_cpp)

sources = [
    str(binding_path),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef_old.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef_ref.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef.cu'),
]

ext = load(
    name=f'fdg_intersect_jit_{os.getpid()}',
    sources=sources,
    extra_include_paths=[
        str(ROOT / 'src/convert'),
        str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu'),
        str(ROOT / 'third_party/eigen'),
    ],
    extra_cflags=['-O3', '-std=c++17'],
    extra_cuda_cflags=['-O3', '-std=c++17'],
    with_cuda=True,
    verbose=True,
)
print('JIT extension loaded:', ext)

Using /home/quanta/.cache/torch_extensions/py310_cu124 as PyTorch extensions root...
Creating extension directory /home/quanta/.cache/torch_extensions/py310_cu124/fdg_intersect_jit_750581...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/quanta/.cache/torch_extensions/py310_cu124/fdg_intersect_jit_750581/build.ninja...
/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fdg_intersect_jit_750581...
Using envvar MAX_JOBS (32) as the number of workers...


[1/5] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fdg_intersect_jit_750581 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert/mesh_to_flexible_dual_grid_gpu -I/mnt/nvmefs/Projects/o-voxel-gpu/third_party/eigen -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -O3 -std=c++17 -c /tmp/fdg_intersect_jit_y2kuov9c/bi

Loading extension module fdg_intersect_jit_750581...


## Load `notebooks/test.glb`, build original and subdivided triangle inputs


In [3]:
import numpy as np
import trimesh


def subdivide_mesh_gpu(verts: torch.Tensor, faces: torch.Tensor, area_threshold: float, max_iters: int):
    from pytorch3d.structures import Meshes

    device = verts.device
    cur = Meshes(verts=[verts], faces=[faces])

    for _ in range(max_iters):
        cv = cur.verts_packed()
        cf = cur.faces_packed()
        edges = cur.edges_packed()
        f2e = cur.faces_packed_to_edges_packed()

        areas = cur.faces_areas_packed()
        large = areas > area_threshold
        if not large.any():
            break

        edge_len = torch.norm(cv[edges[:, 0]] - cv[edges[:, 1]], dim=1)
        _, local_max = torch.max(edge_len[f2e], dim=1)
        max_edge_ids = f2e[torch.arange(cf.shape[0], device=device), local_max]

        target = torch.zeros(edges.shape[0], dtype=torch.bool, device=device)
        target[max_edge_ids[large]] = True
        if not target.any():
            break

        mid = (cv[edges[target, 0]] + cv[edges[target, 1]]) * 0.5
        new_verts = torch.cat([cv, mid], dim=0)
        n_old = cv.shape[0]

        e2new = torch.zeros(edges.shape[0], dtype=torch.long, device=device)
        e2new[target] = torch.arange(n_old, n_old + target.sum().item(), device=device)

        split_counts = target[f2e].sum(dim=1)

        v0, v1, v2 = cf[:, 0], cf[:, 1], cf[:, 2]
        p01_a = torch.minimum(v0, v1)
        p01_b = torch.maximum(v0, v1)
        p12_a = torch.minimum(v1, v2)
        p12_b = torch.maximum(v1, v2)
        p20_a = torch.minimum(v2, v0)
        p20_b = torch.maximum(v2, v0)

        e_act = edges[f2e]
        match01 = (e_act[:, :, 0] == p01_a.unsqueeze(1)) & (e_act[:, :, 1] == p01_b.unsqueeze(1))
        match12 = (e_act[:, :, 0] == p12_a.unsqueeze(1)) & (e_act[:, :, 1] == p12_b.unsqueeze(1))
        match20 = (e_act[:, :, 0] == p20_a.unsqueeze(1)) & (e_act[:, :, 1] == p20_b.unsqueeze(1))

        col01 = match01.long().argmax(dim=1)
        col12 = match12.long().argmax(dim=1)
        col20 = match20.long().argmax(dim=1)

        keep = cf[split_counts == 0]
        out = [keep]

        m1 = split_counts == 1
        if m1.any():
            f1 = cf[m1]
            fe1 = f2e[m1]
            M = f1.shape[0]
            v0_1, v1_1, v2_1 = f1[:, 0], f1[:, 1], f1[:, 2]

            split_col = target[fe1].long().argmax(dim=1)
            c01_m1 = col01[m1]
            c12_m1 = col12[m1]
            c20_m1 = col20[m1]
            is_split_01 = split_col == c01_m1
            is_split_12 = split_col == c12_m1
            is_split_20 = split_col == c20_m1

            split_eid = fe1[torch.arange(M, device=device), split_col]
            vn = e2new[split_eid]

            s1 = torch.zeros(M, 3, dtype=torch.long, device=device)
            s2 = torch.zeros(M, 3, dtype=torch.long, device=device)

            s1[is_split_01] = torch.stack([v0_1[is_split_01], vn[is_split_01], v2_1[is_split_01]], dim=1)
            s2[is_split_01] = torch.stack([vn[is_split_01], v1_1[is_split_01], v2_1[is_split_01]], dim=1)
            s1[is_split_12] = torch.stack([v0_1[is_split_12], v1_1[is_split_12], vn[is_split_12]], dim=1)
            s2[is_split_12] = torch.stack([v0_1[is_split_12], vn[is_split_12], v2_1[is_split_12]], dim=1)
            s1[is_split_20] = torch.stack([v0_1[is_split_20], v1_1[is_split_20], vn[is_split_20]], dim=1)
            s2[is_split_20] = torch.stack([vn[is_split_20], v1_1[is_split_20], v2_1[is_split_20]], dim=1)

            out.extend([s1, s2])

        mm = split_counts >= 2
        if mm.any():
            fm = cf[mm]
            fem = f2e[mm]
            M2 = fm.shape[0]
            v0_m, v1_m, v2_m = fm[:, 0], fm[:, 1], fm[:, 2]

            c01_m = col01[mm]
            c12_m = col12[mm]
            c20_m = col20[mm]

            eid01 = fem[torch.arange(M2, device=device), c01_m]
            e01_is_split = target[eid01]
            e01_v = torch.where(e01_is_split, e2new[eid01], v0_m)

            eid12 = fem[torch.arange(M2, device=device), c12_m]
            e12_is_split = target[eid12]
            e12_v = torch.where(e12_is_split, e2new[eid12], v1_m)

            eid20 = fem[torch.arange(M2, device=device), c20_m]
            e20_is_split = target[eid20]
            e20_v = torch.where(e20_is_split, e2new[eid20], v2_m)

            mf1 = torch.stack([v0_m, e01_v, e20_v], dim=1)
            mf2 = torch.stack([e01_v, v1_m, e12_v], dim=1)
            mf3 = torch.stack([e20_v, e12_v, v2_m], dim=1)
            mf4 = torch.stack([e01_v, e12_v, e20_v], dim=1)
            out.extend([mf1, mf2, mf3, mf4])

        cur = Meshes(verts=[new_verts], faces=[torch.cat(out, dim=0)])

    return cur.verts_packed(), cur.faces_packed()


def triangle_mb(triangles: torch.Tensor) -> float:
    return triangles.numel() * triangles.element_size() / 1024**2


glb_path = ROOT / 'notebooks/test.glb'
loaded = trimesh.load(glb_path, force='scene')
if isinstance(loaded, trimesh.Scene):
    meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    mesh = trimesh.util.concatenate(meshes)
else:
    mesh = loaded
assert isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0

vertices_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)
vmin = vertices_np.min(axis=0)
vmax = vertices_np.max(axis=0)
center = (vmin + vmax) * 0.5
scale = np.float32(0.999 / max((vmax - vmin).max(), 1e-12))
vertices_np = (vertices_np - center) * scale + np.float32(0.5)

vertices_cpu = torch.from_numpy(vertices_np).contiguous()
faces_cpu = torch.from_numpy(faces_np).contiguous()
triangles_cpu = vertices_cpu[faces_cpu].reshape(-1, 9).contiguous()
triangles_cuda = triangles_cpu.to(DEVICE, non_blocking=True).contiguous()

voxel_size_cpu = torch.tensor([1.0 / GRID_SIZE, 1.0 / GRID_SIZE, 1.0 / GRID_SIZE], dtype=torch.float32)
grid_range_cpu = torch.tensor([0, 0, 0, GRID_SIZE, GRID_SIZE, GRID_SIZE], dtype=torch.int32)

print('building subdivided input...')
torch.cuda.empty_cache()
torch.cuda.synchronize()
start = time.perf_counter()
sub_vertices_cuda, sub_faces_cuda = subdivide_mesh_gpu(
    vertices_cpu.to(DEVICE, non_blocking=True).contiguous(),
    faces_cpu.to(DEVICE, non_blocking=True).contiguous(),
    area_threshold=SUBDIVIDE_AREA_THRESHOLD,
    max_iters=SUBDIVIDE_MAX_ITERS,
)
sub_triangles_cuda = sub_vertices_cuda[sub_faces_cuda].reshape(-1, 9).contiguous()
torch.cuda.synchronize()
subdivide_ms = (time.perf_counter() - start) * 1000.0
sub_triangles_cpu = sub_triangles_cuda.detach().cpu().contiguous()
sub_vertices_shape = tuple(sub_vertices_cuda.shape)
sub_faces_shape = tuple(sub_faces_cuda.shape)
del sub_vertices_cuda, sub_faces_cuda
torch.cuda.empty_cache()

INPUT_CASES = {
    'original': {
        'triangles_cpu': triangles_cpu,
        'triangles_cuda': triangles_cuda,
        'vertices_shape': tuple(vertices_cpu.shape),
        'faces_shape': tuple(faces_cpu.shape),
        'triangles': int(triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(triangles_cuda),
        'preprocess_ms': 0.0,
    },
    'subdivided': {
        'triangles_cpu': sub_triangles_cpu,
        'triangles_cuda': sub_triangles_cuda,
        'vertices_shape': sub_vertices_shape,
        'faces_shape': sub_faces_shape,
        'triangles': int(sub_triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(sub_triangles_cuda),
        'preprocess_ms': subdivide_ms,
    },
}

for name, case in INPUT_CASES.items():
    print(
        name,
        'vertices:', case['vertices_shape'],
        'faces:', case['faces_shape'],
        'triangles:', case['triangles'],
        'triangle_mb:', case['triangle_mb'],
        'preprocess_ms:', case['preprocess_ms'],
    )
print('face multiplier:', float(INPUT_CASES['subdivided']['triangles'] / max(INPUT_CASES['original']['triangles'], 1)))
print('voxel_size:', voxel_size_cpu.tolist(), 'grid_range:', grid_range_cpu.tolist())


building subdivided input...
original vertices: (268018, 3) faces: (280333, 3) triangles: 280333 triangle_mb: 9.624469757080078 preprocess_ms: 0.0
subdivided vertices: (3608864, 3) faces: (10304077, 3) triangles: 10304077 triangle_mb: 353.7624092102051 preprocess_ms: 807.3289110034239
face multiplier: 36.75656094715927
voxel_size: [0.001953125, 0.001953125, 0.001953125] grid_range: [0, 0, 0, 512, 512, 512]


Important: the GPU functions take CUDA `triangles`, but `voxel_size` and `grid_range` are passed as CPU tensors because the current C++ reads those scalar tensors on the host.

## Correctness helpers

In [4]:
from typing import NamedTuple


class QEFOutput(NamedTuple):
    voxels: torch.Tensor
    mean_sum: torch.Tensor
    cnt: torch.Tensor
    intersected: torch.Tensor
    qefs: torch.Tensor


class QEFIndexedOutput(NamedTuple):
    voxels: torch.Tensor
    mean_sum: torch.Tensor
    cnt: torch.Tensor
    intersected: torch.Tensor
    qefs: torch.Tensor
    brick_hash_keys: torch.Tensor
    brick_hash_vals: torch.Tensor
    brick_bits: torch.Tensor
    brick_base: torch.Tensor


def as_qef_output(output):
    if isinstance(output, QEFOutput):
        return output
    if isinstance(output, QEFIndexedOutput):
        return QEFOutput(*output[:5])
    return QEFOutput(*output[:5])


def as_indexed_qef_output(output):
    if isinstance(output, QEFIndexedOutput):
        return output
    assert len(output) == 9, f'expected 9 tensors from new intersect_qef, got {len(output)}'
    return QEFIndexedOutput(*output)


def intersect_qef_cpu_py(triangles, voxel_size, grid_range, chunk_triangles):
    return as_qef_output(ext.intersect_qef_cpu(triangles, voxel_size, grid_range, chunk_triangles))


def intersect_qef_old_py(triangles, voxel_size, grid_range, chunk_triangles):
    return as_qef_output(ext.intersect_qef_old(triangles, voxel_size, grid_range, chunk_triangles))


def intersect_qef_ref_py(triangles, voxel_size, grid_range, chunk_triangles):
    return as_qef_output(ext.intersect_qef_ref(triangles, voxel_size, grid_range, chunk_triangles))


def intersect_qef_new_py(triangles, voxel_size, grid_range, chunk_triangles):
    return as_indexed_qef_output(ext.intersect_qef(triangles, voxel_size, grid_range, chunk_triangles))


def voxel_key(voxels: torch.Tensor) -> torch.Tensor:
    voxels = voxels.to(torch.int64)
    rel = voxels - grid_range_cpu[:3].to(torch.int64)
    size = (grid_range_cpu[3:] - grid_range_cpu[:3]).to(torch.int64)
    return rel[:, 0] + size[0] * (rel[:, 1] + size[1] * rel[:, 2])


def canonicalize_qef(output):
    output = as_qef_output(output)
    voxels = output.voxels.detach().cpu().contiguous()
    mean_sum = output.mean_sum.detach().cpu().float().contiguous()
    cnt = output.cnt.detach().cpu().float().contiguous()
    intersected = output.intersected.detach().cpu().bool().contiguous()
    qefs = output.qefs.detach().cpu().float().contiguous()
    keys = voxel_key(voxels)
    order = torch.argsort(keys)
    keys = keys[order]
    unique = torch.unique_consecutive(keys).numel() == keys.numel()
    return {
        'keys': keys,
        'voxels': voxels[order],
        'mean_sum': mean_sum[order],
        'cnt': cnt[order],
        'intersected': intersected[order],
        'qefs': qefs[order],
        'unique': bool(unique),
    }


def canonicalize_occ(voxels):
    voxels = voxels.detach().cpu().contiguous()
    keys = voxel_key(voxels)
    order = torch.argsort(keys)
    return {'keys': keys[order], 'voxels': voxels[order]}


def compare_occ(base, other):
    a = base['keys']
    b = other['keys']
    equal = torch.equal(a, b)
    if equal:
        return {'equal': True, 'count_a': int(a.numel()), 'count_b': int(b.numel()), 'missing': 0, 'extra': 0}
    aset = set(a.tolist())
    bset = set(b.tolist())
    return {
        'equal': False,
        'count_a': int(a.numel()),
        'count_b': int(b.numel()),
        'missing': len(aset - bset),
        'extra': len(bset - aset),
        'missing_sample': sorted(list(aset - bset))[:8],
        'extra_sample': sorted(list(bset - aset))[:8],
    }


def qef_error_metrics(cpu, other):
    occ = compare_occ(cpu, other)
    if not occ['equal']:
        return {'occ_equal': False, **occ}
    rows = {'occ_equal': True, 'count': int(cpu['keys'].numel())}
    for name in ['mean_sum', 'cnt', 'qefs']:
        diff = (other[name] - cpu[name]).float()
        ref = cpu[name].float()
        rows[f'{name}_max_abs'] = float(diff.abs().max().item()) if diff.numel() else 0.0
        rows[f'{name}_rmse'] = float(torch.sqrt(torch.mean(diff * diff)).item()) if diff.numel() else 0.0
        rows[f'{name}_rel_l2'] = float(torch.linalg.vector_norm(diff).item() / max(torch.linalg.vector_norm(ref).item(), 1e-12)) if diff.numel() else 0.0
    rows['intersected_mismatch'] = int((other['intersected'] != cpu['intersected']).sum().item())
    return rows


def show_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)


## Run four QEF implementations and compare occupancy for both inputs


In [5]:
torch.cuda.empty_cache()
torch.cuda.synchronize()

CASE_CANON = {}
CASE_NEW_QEF_OUTPUTS = {}
occ_rows = []
for input_name, case in INPUT_CASES.items():
    qef_outputs = {
        'cpu': intersect_qef_cpu_py(case['triangles_cpu'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
        'old': intersect_qef_old_py(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
        'ref': intersect_qef_ref_py(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
        'new': intersect_qef_new_py(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
    }
    torch.cuda.synchronize()

    canon = {name: canonicalize_qef(out) for name, out in qef_outputs.items()}
    CASE_CANON[input_name] = canon
    CASE_NEW_QEF_OUTPUTS[input_name] = qef_outputs['new']
    del qef_outputs
    torch.cuda.empty_cache()

    assert canon['cpu']['unique'], f'{input_name} cpu produced duplicate voxel keys'
    for name in ['old', 'ref', 'new']:
        row = {'input': input_name, 'method': name, **compare_occ(canon['cpu'], canon[name]), 'unique': canon[name]['unique']}
        occ_rows.append(row)
        assert canon[name]['unique'], f'{input_name} {name} produced duplicate voxel keys'
        assert row['equal'], f'{input_name} {name} occupancy differs from cpu'

NEW_QEF_ORIGINAL = CASE_NEW_QEF_OUTPUTS['original']
NEW_QEF_SUBDIVIDED = CASE_NEW_QEF_OUTPUTS['subdivided']

show_rows(occ_rows)
print('new original index tensors:', NEW_QEF_ORIGINAL.brick_hash_keys.shape, NEW_QEF_ORIGINAL.brick_hash_vals.shape, NEW_QEF_ORIGINAL.brick_bits.shape, NEW_QEF_ORIGINAL.brick_base.shape)
print('new subdivided index tensors:', NEW_QEF_SUBDIVIDED.brick_hash_keys.shape, NEW_QEF_SUBDIVIDED.brick_hash_vals.shape, NEW_QEF_SUBDIVIDED.brick_bits.shape, NEW_QEF_SUBDIVIDED.brick_base.shape)


,input,method,equal,count_a,count_b,missing,extra,unique
0,original,old,True,4676307,4676307,0,0,True
1,original,ref,True,4676307,4676307,0,0,True
2,original,new,True,4676307,4676307,0,0,True
3,subdivided,old,True,4676306,4676306,0,0,True
4,subdivided,ref,True,4676306,4676306,0,0,True
5,subdivided,new,True,4676306,4676306,0,0,True


new original index tensors: torch.Size([524288]) torch.Size([524288]) torch.Size([262144, 16]) torch.Size([262144])
new subdivided index tensors: torch.Size([524288]) torch.Size([524288]) torch.Size([262144, 16]) torch.Size([262144])


## Direct `intersect_occ` API checks

`ref` currently has no standalone `intersect_occ_ref`; for ref occupancy we use `intersect_qef_ref(...)[0]` above. Old and new direct occ APIs are checked below.

In [6]:
occ_api_rows = []
for input_name, case in INPUT_CASES.items():
    occ_direct = {
        'cpu_occ': canonicalize_occ(ext.intersect_occ_cpu(case['triangles_cpu'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)),
        'old_occ': canonicalize_occ(ext.intersect_occ_old(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)),
        'new_occ': canonicalize_occ(ext.intersect_occ(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)),
    }
    torch.cuda.synchronize()

    for name, occ in occ_direct.items():
        row = {'input': input_name, 'method': name, **compare_occ(CASE_CANON[input_name]['cpu'], occ)}
        occ_api_rows.append(row)
        assert row['equal'], f'{input_name} {name} differs from cpu qef occupancy'

show_rows(occ_api_rows)


,input,method,equal,count_a,count_b,missing,extra
0,original,cpu_occ,True,4676307,4676307,0,0
1,original,old_occ,True,4676307,4676307,0,0
2,original,new_occ,True,4676307,4676307,0,0
3,subdivided,cpu_occ,True,4676306,4676306,0,0
4,subdivided,old_occ,True,4676306,4676306,0,0
5,subdivided,new_occ,True,4676306,4676306,0,0


## QEF error estimates against CPU

In [7]:
qef_rows = []
for input_name, canon in CASE_CANON.items():
    for name in ['old', 'ref', 'new']:
        qef_rows.append({'input': input_name, 'method': name, **qef_error_metrics(canon['cpu'], canon[name])})
show_rows(qef_rows)


,input,method,occ_equal,count,mean_sum_max_abs,mean_sum_rmse,mean_sum_rel_l2,cnt_max_abs,cnt_rmse,cnt_rel_l2,qefs_max_abs,qefs_rmse,qefs_rel_l2,intersected_mismatch
0,original,old,True,4676307,0.000006,8.521194e-08,3.210950e-08,0.0,0.0,0.0,0.000028,1.620905e-07,9.370419e-08,0
1,original,ref,True,4676307,0.000004,7.714541e-08,2.906992e-08,0.0,0.0,0.0,0.000029,1.992666e-07,1.153829e-07,0
2,original,new,True,4676307,0.000004,7.147327e-08,2.693251e-08,0.0,0.0,0.0,0.000029,1.961198e-07,1.135599e-07,0
3,subdivided,old,True,4676306,0.000004,8.617130e-08,3.247102e-08,0.0,0.0,0.0,0.000044,1.601999e-07,9.267934e-08,0
4,subdivided,ref,True,4676306,0.000004,2.511712e-08,9.464647e-09,0.0,0.0,0.0,0.000034,1.695321e-07,9.808577e-08,0
5,subdivided,new,True,4676306,0.000004,2.907797e-08,1.095718e-08,0.0,0.0,0.0,0.000034,1.693769e-07,9.799605e-08,0


## GPU timing and memory: cold vs warm for both inputs


In [8]:
def timed_cuda_call(fn, cold=False):
    if cold:
        torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = fn()
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    peak_alloc = torch.cuda.max_memory_allocated() - base_alloc
    peak_reserved = torch.cuda.max_memory_reserved() - base_reserved
    del out
    torch.cuda.synchronize()
    return {
        'ms': float(ms),
        'peak_alloc_mb': float(peak_alloc / 1024**2),
        'peak_reserved_mb': float(peak_reserved / 1024**2),
    }


def summarize_records(records):
    return {
        'ms_median': statistics.median(r['ms'] for r in records),
        'ms_min': min(r['ms'] for r in records),
        'peak_alloc_mb_median': statistics.median(r['peak_alloc_mb'] for r in records),
        'peak_reserved_mb_median': statistics.median(r['peak_reserved_mb'] for r in records),
    }


WARM_REPEATS = int(os.environ.get('FDG_WARM_REPEATS', '10'))
bench_rows = []
for input_name, case in INPUT_CASES.items():
    triangles_cuda = case['triangles_cuda']
    gpu_qef_methods = {
        'old': lambda triangles_cuda=triangles_cuda: intersect_qef_old_py(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
        'ref': lambda triangles_cuda=triangles_cuda: intersect_qef_ref_py(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
        'new': lambda triangles_cuda=triangles_cuda: intersect_qef_new_py(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
    }
    gpu_occ_methods = {
        'old': lambda triangles_cuda=triangles_cuda: ext.intersect_occ_old(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
        'ref': lambda triangles_cuda=triangles_cuda: intersect_qef_ref_py(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES).voxels,
        'new': lambda triangles_cuda=triangles_cuda: ext.intersect_occ(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
    }

    for family, methods in [('occ', gpu_occ_methods), ('qef', gpu_qef_methods)]:
        for name, fn in methods.items():
            cold = timed_cuda_call(fn, cold=True)
            _ = timed_cuda_call(fn, cold=False)
            warm_records = [timed_cuda_call(fn, cold=False) for _ in range(WARM_REPEATS)]
            warm = summarize_records(warm_records)
            bench_rows.append({
                'input': input_name,
                'triangles': case['triangles'],
                'input_triangle_mb': case['triangle_mb'],
                'family': family,
                'method': name,
                'cold_ms': cold['ms'],
                'cold_peak_alloc_mb': cold['peak_alloc_mb'],
                'cold_peak_reserved_mb': cold['peak_reserved_mb'],
                **warm,
            })

show_rows(bench_rows)


,input,triangles,input_triangle_mb,family,method,cold_ms,cold_peak_alloc_mb,cold_peak_reserved_mb,ms_median,ms_min,peak_alloc_mb_median,peak_reserved_mb_median
0,original,280333,9.624470,occ,old,58.779743,288.528320,8.0,54.283264,53.226719,288.528320,0.0
1,original,280333,9.624470,occ,ref,93.851646,1959.215332,688.0,92.735519,90.808319,1959.215332,0.0
2,original,280333,9.624470,occ,new,10.006528,122.238281,2.0,9.544608,9.168128,122.238281,0.0
3,original,280333,9.624470,qef,old,108.990463,2973.485840,2964.0,109.560818,108.941406,2972.870117,0.0
4,original,280333,9.624470,qef,ref,96.653313,1959.215332,688.0,93.976738,91.566078,1959.215332,0.0
5,original,280333,9.624470,qef,new,19.871744,381.661621,2.0,19.198112,18.679071,381.661621,0.0
6,subdivided,10304077,353.762409,occ,old,127.105759,583.728027,176.0,126.886017,126.458878,583.728027,0.0
7,subdivided,10304077,353.762409,occ,ref,60.098560,12604.638184,12290.0,60.464640,60.173313,12604.638184,0.0
8,subdivided,10304077,353.762409,occ,new,23.605247,1227.884277,508.0,22.171648,22.023169,1227.884277,0.0
9,subdivided,10304077,353.762409,qef,old,201.220093,6558.453613,5086.0,193.717255,190.284836,6555.036621,0.0
